In [1]:
from PIL import Image
import random
import numpy as np
from utils.clipemb import CLIP_emb_from_IMG, CLIP_emb_from_TEXT, CLIP_emb_from_tensor
import torch
from torchvision.transforms.functional import to_pil_image


car_path = '/home/pbl/Desktop/dl/dl_project/data/rw_primitives/person.png'
road_path = '/home/pbl/Desktop/dl/dl_project/data/rw_primitives/umbrella.png'


# Open the images
car_img = Image.open(car_path)
road_img = Image.open(road_path)


/home/pbl/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/pbl/.local/lib/python3.8/site-packages/torch/cuda/__init__.py:83: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at  ../c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [ ]:

# Create a white canvas
canvas = Image.new('RGBA', (1000, 1000), color='white')


new_width = 200
aspect_ratio = car_img.height / car_img.width
new_height = int(new_width * aspect_ratio)
car_img = car_img.resize((new_width, new_height))

car_position = (int((canvas.width - car_img.width) / 2), 300)
canvas.paste(car_img, car_position,car_img)



new_width = 550
aspect_ratio = road_img.height / road_img.width
new_height = int(new_width * aspect_ratio)

road_img = road_img.resize((new_width, new_height))

road_position = (300, 100)
canvas.paste(road_img, road_position,road_img)


# Display the result
canvas.show()
canvas.save('GT3.png')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [3]:
torch.cuda.is_available()

False

In [3]:
prompt = "A person under an umbrella"
text_emb = CLIP_emb_from_TEXT(prompt)
text_emb = text_emb / text_emb.norm(dim=1, keepdim=True)

GT = '/home/pbl/Desktop/dl/dl_project/GT2.png'
GT = Image.open(GT)
GT = GT.resize((300,300))

stack = np.zeros((0, 300, 300, 3), dtype=np.float32)
origninalEmb = CLIP_emb_from_IMG(GT)
simarity = np.array([0])

originalSim = origninalEmb @ text_emb[0]
while simarity.max() < originalSim:
    del stack
    torch.cuda.empty_cache() 
    stack = np.zeros((0, 300, 300, 3), dtype=np.float32)
    for i in range(100):
        canvas = Image.new('RGBA', (300, 300), color='white')
        
        # Resize and paste road_img
        w = random.randint(50, 300)
        aspect_ratio = road_img.height / road_img.width
        h = int(w * aspect_ratio)
        resized_road = road_img.resize((w, h))
        #road_position = (random.randint(0, 300-w), random.randint(0, 300-h))
        road_position = (random.randint(0, 250), random.randint(0, 250))
        road_position = (0, 0)
        #resized_road = resized_road.rotate(random.randint(0, 360), expand=True)
        canvas.paste(resized_road, road_position, resized_road)
        
        # Resize and paste car_img
        w = random.randint(50, 300)
        aspect_ratio = car_img.height / car_img.width
        h = int(w * aspect_ratio)
        resized_car = car_img.resize((w, h))
        car_position = (random.randint(0, 250), random.randint(0, 250))
        #resized_car = resized_car.rotate(random.randint(0, 360), expand=True)
        canvas.paste(resized_car, car_position, resized_car)
        
        # Convert the canvas to a numpy array and remove the alpha channel
        np_img = np.array(canvas)[:,:,:3]
        
        # Stack the image along the new axis
        stack = np.concatenate((stack, np_img[None, ...]), axis=0)

    embs = CLIP_emb_from_tensor(torch.tensor(stack, dtype=torch.float32).permute(0,3,1,2))
    embs = embs / embs.norm(dim=1, keepdim=True)
    simarity = (embs @ text_emb[0]) # similarity on all the embeddings
    print(f'max similarity = {simarity.max()}')
    print(f'original similarity = {origninalEmb @ text_emb[0]}')



imgMax = stack[simarity.argmax()] * 255
imgMax = imgMax.astype(np.uint8)

image = Image.fromarray(imgMax)


print(f'other similaritys = {simarity}')
print(f'max similarity = {simarity.max()}')
print(f'original similarity = {origninalEmb @ text_emb[0]}')


max similarity = 0.2737346589565277
original similarity = tensor([0.3256])
max similarity = 0.2861911654472351
original similarity = tensor([0.3256])
max similarity = 0.2716812491416931
original similarity = tensor([0.3256])
max similarity = 0.27376243472099304
original similarity = tensor([0.3256])
max similarity = 0.2751188576221466
original similarity = tensor([0.3256])


KeyboardInterrupt: 

In [6]:
image = Image.fromarray(255-imgMax)
image.show()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [9]:
imgMax = imgMax[[0,1,2],:,:]
imgMax.shape

(3, 300, 3)

In [14]:
g_band, b_band, r_band = image.split()

# Merge the bands into the correct RGB order
rgb_image = Image.merge("RGB", (g_band, b_band,r_band))
rgb_image.show()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [7]:
imgMax.shape

(3, 300, 3)